In [ ]:
# Validar integridad del catálogo
validation_results = []

print("🔍 Validando integridad del catálogo ETL...\n")

for job_id, job_def in catalog.items():
    errors = []
    
    # Verificar que tienen nombre y descripción
    if not job_def.name:
        errors.append("Sin nombre")
    if not job_def.description:
        errors.append("Sin descripción")
    
    # Verificar que el runner es callable
    if not callable(job_def.runner):
        errors.append("Runner no es callable")
    
    # Verificar timeout válido
    if job_def.timeout_seconds <= 0:
        errors.append(f"Timeout inválido ({job_def.timeout_seconds}s)")
    
    status = "✅" if not errors else "⚠️"
    validation_results.append({
        "job_id": job_id,
        "status": status,
        "errors": ", ".join(errors) if errors else "OK"
    })

df_validation = pd.DataFrame(validation_results)
display(df_validation)

# Resumen
valid_count = len([x for x in validation_results if x["status"] == "✅"])
print(f"\n📊 Resumen: {valid_count}/{len(validation_results)} jobs válidos")
if valid_count == len(validation_results):
    print("✅ Todos los jobs están correctamente configurados")
else:
    print(f"⚠️  {len(validation_results) - valid_count} jobs tienen problemas")

## 4. Validar y Testear Jobs

Verifica la salud y disponibilidad de todos los jobs en el catálogo:

In [ ]:
# Selecciona el job_id para inspeccionar
JOB_TO_INSPECT = "comparar_picos_demanda"

if JOB_TO_INSPECT in catalog:
    job = catalog[JOB_TO_INSPECT]
    print(f"📌 Detalles del Job: {JOB_TO_INSPECT}\n")
    print(f"   Nombre: {job.name}")
    print(f"   Descripción: {job.description}")
    print(f"   Permite ejecución paralela: {job.allows_parallel}")
    print(f"   Soporta parámetros: {job.supports_params}")
    print(f"   Timeout: {job.timeout_seconds} segundos")
    if job.default_params:
        print(f"   Parámetros por defecto: {job.default_params}")
    else:
        print(f"   Parámetros por defecto: Ninguno")
else:
    print(f"❌ El job '{JOB_TO_INSPECT}' no existe en el catálogo")

## 3. Detalles de un Job Específico

Inspecciona los detalles, parámetros por defecto y configuración de un job particular:

In [ ]:
# ============ CONFIGURACIÓN ============
# Selecciona el job_id a ejecutar
JOB_ID = "proyecciones_csv"  # Cambiar según el job a ejecutar

# Parámetros opcionales (solo para jobs que soportan params)
PARAMS = None  # Dejar como None si no se usan parámetros

# Ejemplos de parámetros para jobs específicos:
# "comparar_picos_demanda" -> {"modelo_tika": 485, "semana_actual": 2613}
# "compare_tika_produccion_despachos" -> {"modelos_tika": [161, 328, 485, 405]}
# "quipus_vs_proy" -> {"semana_actual": 2613}

# ============ INICIO DE EJECUCIÓN ============
if JOB_ID not in catalog:
    print(f"❌ Error: El job '{JOB_ID}' no existe en el catálogo.")
else:
    job_def = catalog[JOB_ID]
    print(f"🚀 Ejecutando: {job_def.name}")
    print(f"   ID: {JOB_ID}")
    print(f"   Descripción: {job_def.description}")
    print(f"   Timeout: {job_def.timeout_seconds}s")
    if PARAMS:
        print(f"   Parámetros: {PARAMS}")
    print(f"   Fecha/hora: {datetime.now().isoformat()}")
    print("\n" + "="*80 + "\n")
    
    try:
        start_time = time.time()
        result = job_def.runner(PARAMS)
        elapsed = time.time() - start_time
        
        print(f"\n✅ Job completado exitosamente")
        print(f"⏱️  Tiempo: {elapsed:.2f}s")
        print(f"📁 Resultado: {result}")
        
    except Exception as e:
        print(f"\n❌ Error durante la ejecución:")
        print(f"   {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()

## 2. Ejecutar un Job Específico

Selecciona el `job_id` y ejecuta el job. Opcionalmente, pasa parámetros si el job los soporta.

In [ ]:
# Construir catálogo de jobs
catalog = build_job_catalog()

# Mostrar jobs disponibles
jobs_data = []
for job_id, job_def in catalog.items():
    jobs_data.append({
        "job_id": job_id,
        "name": job_def.name,
        "description": job_def.description,
        "allows_parallel": job_def.allows_parallel,
        "supports_params": job_def.supports_params,
        "timeout_seconds": job_def.timeout_seconds,
    })

df_jobs = pd.DataFrame(jobs_data)
print(f"\n📋 Total de jobs disponibles: {len(df_jobs)}\n")
display(df_jobs)

## 1. Jobs Disponibles

Lista de todos los ETL jobs registrados en el catálogo:

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime
import time

# Add project root to path
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

from app.services.etl_catalog import build_job_catalog

print("✓ Librerías importadas exitosamente")

# Ejecutar ETL Jobs

Este notebook permite ejecutar y administrar los jobs ETL disponibles en el catálogo de la aplicación.